# Level 0: inspect the snapshot

The first look at a snapshot written by s2protocol-rs `write-arrow-ipc` (or swarmy's Optimize):
what is in it, how big it is, what a row looks like and whether the data is sane. Nothing is
aggregated yet, the questions are the ones to ask of any dataset before trusting a chart made from
it.

Every question is answered twice, with [polars](https://docs.pola.rs/) expressions and with
[DuckDB](https://duckdb.org/docs/) SQL over the same files; `s2.same()` checks that both return the
same rows, so each pair is a worked translation between the two.

The snapshot is read from `$S2_IPC_DIR`, or `~/SC2Replays/ipcs`, see `snapshot.py` next to this
notebook. Next: [Level 1](./Level_1_One_Game.ipynb), one game over time.

In [1]:
import polars as pl
import plotly.express as px

import snapshot as s2
import viz

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_width_chars(200)
pl.Config.set_fmt_str_lengths(60)

# The snapshot: `$S2_IPC_DIR`, or `ipcs` in the replay directory when written by swarmy.
IPC_DIR = s2.ipc_dir()
# polars reads the files lazily, one LazyFrame per file.
details = s2.scan(IPC_DIR, "details")
stats = s2.scan(IPC_DIR, "stats")
upgrades = s2.scan(IPC_DIR, "upgrades")
unit_born = s2.scan(IPC_DIR, "unit_born")
unit_died = s2.scan(IPC_DIR, "unit_died")
# DuckDB reads the same files as SQL views of the same names, plus `players` and `games_1v1`.
con = s2.duck(IPC_DIR)

## What was written

`metadata.json` is written with the snapshot: the version of s2protocol-rs, when it was written,
how many replays it holds and the size of every file. The snapshot is partitioned by the day the
games were played, `year=YYYY/month=MM/day=DD/<file>.arrow`, each file is one table.

In [2]:
import json
from pathlib import Path

metadata = json.loads((Path(IPC_DIR).expanduser() / "metadata.json").read_text())
print(
    f"{metadata['replays']} replays, s2protocol-rs {metadata['s2protocol_version']}, written on "
    f"{metadata['created_at'][:10]} in {metadata['elapsed_ms'] / 1000:.0f} s, {metadata['partitions']} days"
)
pl.DataFrame(metadata["files"]).with_columns(
    (pl.col("bytes") / 2**20).round(1).alias("MiB"),
    (pl.col("records") / metadata["replays"]).round(0).alias("records_per_replay"),
)

36394 replays, s2protocol-rs 5.0.2, written on 2026-11-03 in 101 s, 2095 days


name,bytes,records,MiB,records_per_replay
str,i64,i64,f64,f64
"""details.arrow""",28314518,91547,27.0,3.0
"""stats.arrow""",438844694,11487095,418.5,316.0
"""upgrades.arrow""",60569310,3128158,57.8,86.0
"""unit_born.arrow""",2341598214,134224130,2233.1,3688.0
"""unit_died.arrow""",1257117782,93023833,1198.9,2556.0
"""cmd_target_point.arrow""",1157842382,101832511,1104.2,2798.0
"""cmd_target_unit.arrow""",324190638,16206815,309.2,445.0


## How many rows

The tables are scanned lazily and a bare `pl.len()` reads no columns, `collect_all` runs the
seven counts at once. In SQL `count(*)` on the views counts the same files.

In [3]:
counts = pl.collect_all([s2.scan(IPC_DIR, name).select(pl.len()) for name in s2.FILES])
rows = pl.DataFrame({"table": s2.FILES, "rows": [count.item() for count in counts]})
rows

table,rows
str,i64
"""details""",91547
"""stats""",11487095
"""upgrades""",3128158
"""unit_born""",134224130
"""unit_died""",93023833
"""cmd_target_point""",101832511
"""cmd_target_unit""",16206815


In [4]:
sql = con.sql(" UNION ALL ".join(f"SELECT '{name}' AS table, count(*) AS rows FROM {name}" for name in s2.FILES))
s2.same(rows, sql)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

'same 7 rows, 2 columns'

## The columns

`collect_schema()` gives the names and the types without reading the data, `DESCRIBE` is the SQL
twin. The details have one row per player of every game: the name, the race (localized, i.e.
`Terraner`), the result, the map (`title`), the date (`ext_datetime`) and the file. The replay is
identified by `ext_fs_id` in every table.

In [5]:
details.collect_schema()

Schema([('player_name', String),
        ('player_toon_region', UInt8),
        ('player_toon_program_id', UInt32),
        ('player_toon_realm', UInt32),
        ('player_toon_id', UInt64),
        ('player_race', String),
        ('player_color_a', UInt8),
        ('player_color_r', UInt8),
        ('player_color_g', UInt8),
        ('player_color_b', UInt8),
        ('player_control', UInt8),
        ('player_team_id', UInt8),
        ('player_observe', UInt8),
        ('player_result', String),
        ('player_working_set_slot_id', UInt8),
        ('player_hero', String),
        ('title', String),
        ('map_info_sha256', String),
        ('is_blizzard_map', Boolean),
        ('time_utc', Int64),
        ('time_local_offset', Int64),
        ('lobby_slot_user_id', Int64),
        ('lobby_slot_observe', UInt8),
        ('lobby_slot_map_size_x', UInt8),
        ('lobby_slot_map_size_y', UInt8),
        ('lobby_slot_difficulty', Int64),
        ('lobby_slot_ai_build', Int64),
   

In [6]:
con.sql("DESCRIBE details")

┌────────────────────────┬──────────────┬─────────┬─────────┬─────────┬─────────┐
│      column_name       │ column_type  │  null   │   key   │ default │  extra  │
│        varchar         │   varchar    │ varchar │ varchar │ varchar │ varchar │
├────────────────────────┼──────────────┼─────────┼─────────┼─────────┼─────────┤
│ player_name            │ VARCHAR      │ YES     │ NULL    │ NULL    │ NULL    │
│ player_toon_region     │ UTINYINT     │ YES     │ NULL    │ NULL    │ NULL    │
│ player_toon_program_id │ UINTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ player_toon_realm      │ UINTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ player_toon_id         │ UBIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ player_race            │ VARCHAR      │ YES     │ NULL    │ NULL    │ NULL    │
│ player_color_a         │ UTINYINT     │ YES     │ NULL    │ NULL    │ NULL    │
│ player_color_r         │ UTINYINT     │ YES     │ NULL    │ NULL    │ NULL    │
│ player_color_g

## What a row looks like

A few rows of each table, to see the values. `s2.peek()` is `head().collect()` with a fallback
for a polars 1.40 bug on the partitioned scan, see its docstring. The tracker
events (`stats`, `upgrades`, `unit_born`, `unit_died`) identify the player by `player_id`, the
details by name; `s2.players()` maps one to the other, see below.

In [7]:
s2.peek(details.select("ext_fs_id", "ext_datetime", "title", "player_name", "player_race", "player_result", "player_team_id"), 4)

ext_fs_id,ext_datetime,title,player_name,player_race,player_result,player_team_id
u64,datetime[ns],str,str,str,str,u8
0,2019-08-13 22:39:44.775573,"""King's Cove LE""","""&lt;MkersA&gt;<sp/>Rocck""","""Zerg""","""Win""",0
0,2019-08-13 22:39:44.775573,"""King's Cove LE""","""&lt;Mealen&gt;<sp/>Doombringer""","""Zerg""","""Loss""",1
1,2019-08-13 22:47:30.061092,"""Cyber Forest LE""","""&lt;Xanz&gt;<sp/>NoVA""","""Zerg""","""Win""",0
1,2019-08-13 22:47:30.061092,"""Cyber Forest LE""","""&lt;Mealen&gt;<sp/>Doombringer""","""Zerg""","""Loss""",1


In [8]:
con.sql("SELECT ext_fs_id, player_id, ext_replay_loop, minerals_current, vespene_current, workers_active_count, food_used, food_made FROM stats LIMIT 4")

┌───────────┬───────────┬─────────────────┬──────────────────┬─────────────────┬──────────────────────┬───────────┬───────────┐
│ ext_fs_id │ player_id │ ext_replay_loop │ minerals_current │ vespene_current │ workers_active_count │ food_used │ food_made │
│  uint64   │   uint8   │      int64      │      int32       │      int32      │        int32         │   int32   │   int32   │
├───────────┼───────────┼─────────────────┼──────────────────┼─────────────────┼──────────────────────┼───────────┼───────────┤
│         1 │         1 │               1 │               50 │               0 │                   12 │        12 │        14 │
│         1 │         2 │               1 │               50 │               0 │                   12 │        12 │        14 │
│         1 │         1 │             160 │               40 │               0 │                   12 │        13 │        14 │
│         1 │         2 │             160 │               35 │               0 │                   12 │ 

In [9]:
s2.peek(unit_born.select("ext_fs_id", "ext_replay_loop", "control_player_id", "player_name", "unit_type_name", "creator_unit_type_name", "creator_ability_name", "x", "y").filter(pl.col("ext_replay_loop") > 0), 6)

ext_fs_id,ext_replay_loop,control_player_id,player_name,unit_type_name,creator_unit_type_name,creator_ability_name,x,y
u64,i64,u8,str,str,str,str,f32,f32
0,21,2,"""&lt;Mealen&gt;<sp/>Doombringer""","""Egg""","""Larva""",null,31.0,132.0
0,23,1,"""&lt;MkersA&gt;<sp/>Rocck""","""Egg""","""Larva""",null,144.0,21.0
0,261,2,"""&lt;Mealen&gt;<sp/>Doombringer""","""Larva""","""Hatchery""","""""",31.0,132.0
0,263,1,"""&lt;MkersA&gt;<sp/>Rocck""","""Larva""","""Hatchery""","""""",144.0,21.0
0,292,2,"""&lt;Mealen&gt;<sp/>Doombringer""","""Egg""","""Larva""",null,32.0,132.0
0,293,2,"""&lt;Mealen&gt;<sp/>Doombringer""","""Drone""","""Egg""","""LarvaTrain""",31.0,132.0


## The span of the snapshot

How many games, from when to when, on how many maps. `ext_datetime` is the date the game was
played, in UTC.

In [10]:
span = details.select(
    pl.col("ext_fs_id").n_unique().alias("games"),
    pl.len().alias("player_rows"),
    pl.col("ext_datetime").min().alias("first_game"),
    pl.col("ext_datetime").max().alias("last_game"),
    pl.col("ext_datetime").dt.date().n_unique().alias("days_with_games"),
    pl.col("title").n_unique().alias("maps"),
    pl.col("player_name").n_unique().alias("player_names"),
).collect()
span

games,player_rows,first_game,last_game,days_with_games,maps,player_names
u32,u32,datetime[ns],datetime[ns],u32,u32,u32
36394,91547,2019-08-13 22:39:44.775573,2026-10-24 19:55:25.982270,2095,757,17483


In [11]:
sql = con.sql("""
    SELECT
        count(DISTINCT ext_fs_id) AS games,
        count(*) AS player_rows,
        min(ext_datetime) AS first_game,
        max(ext_datetime) AS last_game,
        count(DISTINCT ext_datetime::DATE) AS days_with_games,
        count(DISTINCT title) AS maps,
        count(DISTINCT player_name) AS player_names
    FROM details
""")
s2.same(span, sql)

'same 1 rows, 7 columns'

## The values of a column

`value_counts` of the categorical columns show what to filter on. The race is localized in the
details (the language of the client of the player who saved the replay), `s2.race_letter()` maps
every spelling to `T`, `Z` or `P` (`?` for the others, i.e. the arcade races); the SQL twin joins
the `races` view with the same table of spellings.

In [12]:
races = (
    details.group_by("player_race", s2.race_letter().alias("race"))
    .agg(pl.len().alias("player_rows"))
    .sort("player_rows", descending=True)
    .collect()
)
races

player_race,race,player_rows
str,str,u32
"""Zerg""","""Z""",33043
"""Terran""","""T""",26931
"""Protoss""","""P""",26423
"""프로토스""","""P""",890
"""테란""","""T""",768
"""저그""","""Z""",648
"""星灵""","""P""",594
"""人类""","""T""",467
"""Протоссы""","""P""",281


In [13]:
sql = con.sql("""
    SELECT player_race, coalesce(r.letter, '?') AS race, count(*) AS player_rows
    FROM details d
    LEFT JOIN races r ON r.race = d.player_race
    GROUP BY ALL
    ORDER BY player_rows DESC
""")
s2.same(races, sql)

'same 55 rows, 3 columns'

In [14]:
details.group_by("player_result").agg(pl.len().alias("player_rows")).sort("player_rows", descending=True).collect()

player_result,player_rows
str,u32
"""Win""",42536
"""Loss""",41216
"""Undecided""",7722
"""Tie""",73


## The shape of the games

How many players, observers and computers each game has: a 1v1 is two players and no computer,
a game against the A.I. has `player_control == 3` rows. A game with observers has more rows in
the details than players. The group of the groups: first per game, then per shape.

In [15]:
shapes = (
    details.group_by("ext_fs_id")
    .agg(
        (pl.col("player_observe") == 0).sum().alias("players"),
        (pl.col("player_observe") != 0).sum().alias("observers"),
        (pl.col("player_control") == 3).sum().alias("computers"),
        pl.col("player_team_id").n_unique().alias("teams"),
    )
    .group_by("players", "observers", "computers", "teams")
    .agg(pl.len().alias("games"))
    .sort("games", descending=True)
    .collect()
)
shapes

players,observers,computers,teams,games
u32,u32,u32,u32,u32
2,0,0,2,25875
4,0,2,2,3571
4,0,0,2,2065
2,0,1,2,1922
3,0,2,2,1696
6,0,0,2,239
6,0,4,2,233
10,0,0,1,141
7,0,5,2,134


In [16]:
sql = con.sql("""
    WITH per_game AS (
        SELECT
            ext_fs_id,
            count(*) FILTER (WHERE player_observe = 0) AS players,
            count(*) FILTER (WHERE player_observe <> 0) AS observers,
            count(*) FILTER (WHERE player_control = 3) AS computers,
            count(DISTINCT player_team_id) AS teams
        FROM details
        GROUP BY ext_fs_id
    )
    SELECT players, observers, computers, teams, count(*) AS games
    FROM per_game
    GROUP BY ALL
    ORDER BY games DESC
""")
s2.same(shapes, sql)

'same 74 rows, 5 columns'

## The ranges of the numbers

`describe()` and DuckDB's `SUMMARIZE` give the minimum, the maximum, the quartiles and the nulls
of every column at once. The stats are sampled every 160 game loops (about 7 seconds at Faster
speed); `food_used` is the supply, `workers_active_count` the workers mining, the collection rates
are per minute.

In [17]:
STATS_COLUMNS = ["ext_replay_loop", "workers_active_count", "minerals_collection_rate", "vespene_collection_rate", "minerals_current", "food_used", "food_made"]
stats.select(STATS_COLUMNS).collect().describe()

statistic,ext_replay_loop,workers_active_count,minerals_collection_rate,vespene_collection_rate,minerals_current,food_used,food_made
str,f64,f64,f64,f64,f64,f64,f64
"""count""",1.1487095e7,1.1487095e7,1.1487095e7,1.1487095e7,1.1487095e7,1.1487095e7,1.1487095e7
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",19419.592424,38.178844,1491.476389,486.889068,7866.686273,90.869616,127.424473
"""std""",27731.087441,23.320384,1087.436226,549.223289,997127.364669,65.748722,228.354541
"""min""",0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""25%""",5280.0,19.0,783.0,67.0,133.0,31.0,46.0
"""50%""",11040.0,35.0,1343.0,335.0,365.0,78.0,109.0
"""75%""",20000.0,58.0,2155.0,694.0,1400.0,148.0,196.0
"""max""",524287.0,200.0,159028.0,72794.0,2.68435651e8,907.0,10284.0


In [18]:
con.sql(f"SUMMARIZE SELECT {', '.join(STATS_COLUMNS)} FROM stats").pl().select("column_name", "min", "max", "q25", "q50", "q75", "avg", "null_percentage")

column_name,min,max,q25,q50,q75,avg,null_percentage
str,str,str,str,str,str,str,"decimal[9,2]"
"""ext_replay_loop""","""0""","""524287""","""5213""","""11069""","""20028""","""19419.59242419428""",0.00
"""workers_active_count""","""0""","""200""","""19""","""35""","""58""","""38.178843998417356""",0.00
"""minerals_collection_rate""","""0""","""159028""","""779""","""1343""","""2154""","""1491.4763885908492""",0.00
"""vespene_collection_rate""","""0""","""72794""","""72""","""335""","""706""","""486.8890681238381""",0.00
"""minerals_current""","""0""","""268435651""","""133""","""366""","""1405""","""7866.686272726046""",0.00
"""food_used""","""0""","""907""","""31""","""78""","""148""","""90.8696164696122""",0.00
"""food_made""","""0""","""10284""","""45""","""108""","""196""","""127.42447311526543""",0.00


## Sanity checks

Cheap checks that catch most wrong charts later: the replay ids are unique, every game has one
winner and one loser, every game is dated, and the tracker `player_id` of both players was found
(`s2.players()` maps the player names of the details to the ids of the events through the
`unit_born` events, the computers have no name there).

In [19]:
checks = pl.concat(
    [
        details.select(
            pl.lit("one id per replay file").alias("check"),
            (pl.col("ext_fs_id").n_unique() == pl.col("ext_fs_sha256").n_unique()).alias("ok"),
        ),
        details.select(pl.lit("every game dated").alias("check"), pl.col("ext_datetime").is_not_null().all().alias("ok")),
        details.filter(pl.col("player_observe") == 0)
        .group_by("ext_fs_id")
        .agg((pl.col("player_result") == "Win").sum().alias("wins"), (pl.col("player_result") == "Loss").sum().alias("losses"))
        .select(pl.lit("one winner and one loser per game").alias("check"), ((pl.col("wins") == 1) & (pl.col("losses") >= 1)).all().alias("ok")),
        s2.players(IPC_DIR)
        .group_by("ext_fs_id")
        .agg(pl.len().alias("mapped"))
        .join(details.group_by("ext_fs_id").agg((pl.col("player_observe") == 0).sum().alias("players")), on="ext_fs_id", how="right")
        .select(pl.lit("every player mapped to a tracker id").alias("check"), (pl.col("mapped") == pl.col("players")).all().alias("ok")),
    ]
).collect()
checks

check,ok
str,bool
"""one id per replay file""",true
"""every game dated""",true
"""one winner and one loser per game""",false
"""every player mapped to a tracker id""",false


In [20]:
sql = con.sql("""
    SELECT 'one id per replay file' AS check, count(DISTINCT ext_fs_id) = count(DISTINCT ext_fs_sha256) AS ok FROM details
    UNION ALL
    SELECT 'every game dated', bool_and(ext_datetime IS NOT NULL) FROM details
    UNION ALL
    SELECT 'one winner and one loser per game', bool_and(wins = 1 AND losses >= 1)
    FROM (
        SELECT ext_fs_id, count(*) FILTER (WHERE player_result = 'Win') AS wins, count(*) FILTER (WHERE player_result = 'Loss') AS losses
        FROM details WHERE player_observe = 0 GROUP BY ext_fs_id
    )
    UNION ALL
    SELECT 'every player mapped to a tracker id', bool_and(coalesce(mapped, 0) = players)
    FROM (SELECT ext_fs_id, count(*) FILTER (WHERE player_observe = 0) AS players FROM details GROUP BY ext_fs_id) d
    LEFT JOIN (SELECT ext_fs_id, count(*) AS mapped FROM players GROUP BY ext_fs_id) p USING (ext_fs_id)
""")
s2.same(checks, sql)

'same 4 rows, 2 columns'

A check that fails is not an error of the notebook, it is a fact about the data to remember: a
snapshot of games against the A.I. has no tracker id for the computers, a snapshot with
disconnects has games without a winner, and the queries below them must say what they do with
those rows.

## How long the games last

The first measure of every game: the last stats sample of the tracker, in game seconds at Faster
speed (22.4 game loops per second, `s2.game_seconds()`, the `game_seconds` macro in SQL).

In [21]:
durations = (
    stats.group_by("ext_fs_id")
    .agg(pl.col("ext_replay_loop").max().alias("last_loop"))
    .with_columns((s2.game_seconds("last_loop") / 60).alias("minutes"))
    .select("ext_fs_id", "minutes")
    .collect()
)
durations.select(
    pl.col("minutes").min().round(1).alias("shortest"),
    pl.col("minutes").median().round(1).alias("median"),
    pl.col("minutes").max().round(1).alias("longest"),
    pl.len().alias("games"),
)

shortest,median,longest,games
f64,f64,f64,u32
0.0,11.9,390.1,36392


In [22]:
sql = con.sql("SELECT ext_fs_id, game_seconds(max(ext_replay_loop)) / 60 AS minutes FROM stats GROUP BY ext_fs_id")
s2.same(durations, sql)

'same 36392 rows, 2 columns'

In [23]:
fig = px.histogram(durations, x="minutes", nbins=60, title="How long the games last (minutes)", labels={"minutes": "game length, minutes"})
fig.update_layout(yaxis_title="games", bargap=0.05)
viz.show(fig, "level_0_game_length")

## What to take from Level 0

- The snapshot is a set of tables keyed by `ext_fs_id`, the tracker tables identify the players
  by `player_id` and `s2.players()` / the `players` view map them to the details.
- `collect_schema()`, `head()`, `value_counts`, `describe()` and the sanity checks come before
  any aggregation; the facts they show (observers, computers, undecided games) are the filters of
  the later levels.
- polars and DuckDB read the same files, every query here has a twin: write the one that reads
  better for the question.

Next: [Level 1](./Level_1_One_Game.ipynb) follows one game from the first worker to the last fight.